# Llama-3.2-1B - Structured Output Benchmark v2 - Phase 2 (Temperature Probe)
- **Model**: `meta-llama/Llama-3.2-1B-Instruct`
- **Parameters**: 1B (dense)
- **Hardware**: RTX 4090
- **Phase 2 (probe)**: 14 tasks x 3 temperatures (0.0, 0.3, 0.7) x 3 stochastic samples
- **Goal**: Determine whether Phase-1 failures are deterministic (systematic) or sampling-induced (stochastic). T=0.0 re-confirms Phase 1; T>0.0 tests sampling robustness.
- **Special handling**: set `pad_token_id=128001` in BASE_GENERATE_KWARGS; standard chat template (no /no_think)
- **Scope**: SUPPORTING experiment (not the paper's headline). Llama 1B is a failing model of interest (Phase-1 scaling-inversion). If sampling reveals new failure modes absent at T=0, expand to 5 temps x 5 samples.


In [ ]:
!pip install huggingface_hub

from huggingface_hub import login
login()

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import torch
import json
import time
import re
from datetime import datetime

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


PyTorch: 2.10.0
CUDA available: False


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Model: {MODEL_NAME}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/Users/akash/miniconda3/envs/ddods/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0607 13:53:39.485000 94468 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
"""
Task definitions for structured output benchmark.

Each task has:
- name: task identifier
- prompt: the actual prompt to send to the model
- schema: the expected JSON schema for valid output
- evaluator: how to check correctness
"""

import json

# ============================================================
# TASK CATEGORY 1: Simple JSON Generation
# Generate a JSON object from a natural language description
# ============================================================

SIMPLE_JSON_TASKS = [
    {
        "id": "json_simple_person",
        "category": "json_generation",
        "difficulty": "easy",
        "prompt": "Generate a JSON object for a person with the following fields: name (string), age (number), email (string), and city (string). Use realistic values.",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "number"},
                "email": {"type": "string"},
                "city": {"type": "string"}
            },
            "required": ["name", "age", "email", "city"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_simple_product",
        "category": "json_generation",
        "difficulty": "easy",
        "prompt": "Create a JSON object for a product listing with: product_name (string), price (number), in_stock (boolean), and category (string).",
        "schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string"},
                "price": {"type": "number"},
                "in_stock": {"type": "boolean"},
                "category": {"type": "string"}
            },
            "required": ["product_name", "price", "in_stock", "category"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_nested_address",
        "category": "json_generation",
        "difficulty": "medium",
        "prompt": "Generate a JSON object for a user profile. It must have: name (string), age (number), and address (object with street, city, state, zip).",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "number"},
                "address": {
                    "type": "object",
                    "properties": {
                        "street": {"type": "string"},
                        "city": {"type": "string"},
                        "state": {"type": "string"},
                        "zip": {"type": "string"}
                    },
                    "required": ["street", "city", "state", "zip"],
                    "additionalProperties": False
                }
            },
            "required": ["name", "age", "address"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_array_orders",
        "category": "json_generation",
        "difficulty": "medium",
        "prompt": "Create a JSON array containing 3 order objects. Each order should have: order_id (string), items (array of strings), total (number), and status (string that must be one of: pending, shipped, delivered).",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "items": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "total": {"type": "number"},
                    "status": {"type": "string", "enum": ["pending", "shipped", "delivered"]}
                },
                "required": ["order_id", "items", "total", "status"],
                "additionalProperties": False
            },
            "minItems": 3,
            "maxItems": 3
        }
    },
    {
        "id": "json_complex_api",
        "category": "json_generation",
        "difficulty": "hard",
        "prompt": "Generate a JSON object representing an API response. It should have: status (number), message (string), data (object with users array, where each user has id, name, email, role where role is one of admin/user/moderator), and metadata (object with total_count, page, per_page).",
        "schema": {
            "type": "object",
            "properties": {
                "status": {"type": "number"},
                "message": {"type": "string"},
                "data": {
                    "type": "object",
                    "properties": {
                        "users": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "id": {"type": "number"},
                                    "name": {"type": "string"},
                                    "email": {"type": "string"},
                                    "role": {"type": "string", "enum": ["admin", "user", "moderator"]}
                                },
                                "required": ["id", "name", "email", "role"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["users"],
                    "additionalProperties": False
                },
                "metadata": {
                    "type": "object",
                    "properties": {
                        "total_count": {"type": "number"},
                        "page": {"type": "number"},
                        "per_page": {"type": "number"}
                    },
                    "required": ["total_count", "page", "per_page"],
                    "additionalProperties": False
                }
            },
            "required": ["status", "message", "data", "metadata"],
            "additionalProperties": False
        }
    }
]

# ============================================================
# TASK CATEGORY 2: Schema Adherence
# Given a schema, generate output that matches it
# ============================================================

SCHEMA_ADHERENCE_TASKS = [
    {
        "id": "schema_weather",
        "category": "schema_adherence",
        "difficulty": "easy",
        "prompt": "Output a valid JSON object matching this exact schema. Generate realistic weather data:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"location\":{\"type\":\"string\"},\"temperature\":{\"type\":\"number\"},\"unit\":{\"type\":\"string\",\"enum\":[\"celsius\",\"fahrenheit\"]},\"conditions\":{\"type\":\"string\"},\"humidity\":{\"type\":\"number\",\"minimum\":0,\"maximum\":100}},\"required\":[\"location\",\"temperature\",\"unit\",\"conditions\",\"humidity\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
                "temperature": {"type": "number"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                "conditions": {"type": "string"},
                "humidity": {"type": "number", "minimum": 0, "maximum": 100}
            },
            "required": ["location", "temperature", "unit", "conditions", "humidity"],
            "additionalProperties": False
        }
    },
    {
        "id": "schema_database_record",
        "category": "schema_adherence",
        "difficulty": "medium",
        "prompt": "Generate a JSON object matching this schema representing a database record. Fill in realistic values:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"id\":{\"type\":\"string\",\"pattern\":\"^[a-f0-9]{8}$\"},\"created_at\":{\"type\":\"string\",\"format\":\"date-time\"},\"type\":{\"type\":\"string\",\"enum\":[\"customer\",\"vendor\",\"employee\"]},\"active\":{\"type\":\"boolean\"},\"tags\":{\"type\":\"array\",\"items\":{\"type\":\"string\"},\"maxItems\":5}},\"required\":[\"id\",\"created_at\",\"type\",\"active\",\"tags\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "id": {"type": "string", "pattern": "^[a-f0-9]{8}$"},
                "created_at": {"type": "string", "format": "date-time"},
                "type": {"type": "string", "enum": ["customer", "vendor", "employee"]},
                "active": {"type": "boolean"},
                "tags": {
                    "type": "array",
                    "items": {"type": "string"},
                    "maxItems": 5
                }
            },
            "required": ["id", "created_at", "type", "active", "tags"],
            "additionalProperties": False
        }
    },
    {
        "id": "schema_config_file",
        "category": "schema_adherence",
        "difficulty": "hard",
        "prompt": "Generate a valid JSON configuration object matching this schema for a web server config:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"server\":{\"type\":\"object\",\"properties\":{\"host\":{\"type\":\"string\"},\"port\":{\"type\":\"integer\",\"minimum\":1,\"maximum\":65535},\"ssl\":{\"type\":\"object\",\"properties\":{\"enabled\":{\"type\":\"boolean\"},\"cert_path\":{\"type\":\"string\"},\"key_path\":{\"type\":\"string\"}},\"required\":[\"enabled\"]}},\"required\":[\"host\",\"port\",\"ssl\"]},\"logging\":{\"type\":\"object\",\"properties\":{\"level\":{\"type\":\"string\",\"enum\":[\"debug\",\"info\",\"warn\",\"error\"]},\"file\":{\"type\":\"string\"},\"rotate\":{\"type\":\"boolean\"}},\"required\":[\"level\"]},\"cors\":{\"type\":\"object\",\"properties\":{\"enabled\":{\"type\":\"boolean\"},\"origins\":{\"type\":\"array\",\"items\":{\"type\":\"string\"}},\"methods\":{\"type\":\"array\",\"items\":{\"type\":\"string\",\"enum\":[\"GET\",\"POST\",\"PUT\",\"DELETE\",\"PATCH\"]}}},\"required\":[\"enabled\"]}},\"required\":[\"server\",\"logging\",\"cors\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "server": {
                    "type": "object",
                    "properties": {
                        "host": {"type": "string"},
                        "port": {"type": "integer"},
                        "ssl": {
                            "type": "object",
                            "properties": {
                                "enabled": {"type": "boolean"},
                                "cert_path": {"type": "string"},
                                "key_path": {"type": "string"}
                            },
                            "required": ["enabled"],
                            "additionalProperties": False
                        }
                    },
                    "required": ["host", "port", "ssl"],
                    "additionalProperties": False
                },
                "logging": {
                    "type": "object",
                    "properties": {
                        "level": {"type": "string", "enum": ["debug", "info", "warn", "error"]},
                        "file": {"type": "string"},
                        "rotate": {"type": "boolean"}
                    },
                    "required": ["level"],
                    "additionalProperties": False
                },
                "cors": {
                    "type": "object",
                    "properties": {
                        "enabled": {"type": "boolean"},
                        "origins": {"type": "array", "items": {"type": "string"}},
                        "methods": {
                            "type": "array",
                            "items": {"type": "string", "enum": ["GET", "POST", "PUT", "DELETE", "PATCH"]}
                        }
                    },
                    "required": ["enabled"],
                    "additionalProperties": False
                }
            },
            "required": ["server", "logging", "cors"],
            "additionalProperties": False
        }
    }
]

# ============================================================
# TASK CATEGORY 3: Function Calling Format
# Generate tool/function call format (OpenAI-style)
# ============================================================

FUNCTION_CALLING_TASKS = [
    {
        "id": "funcall_get_weather",
        "category": "function_calling",
        "difficulty": "easy",
        "prompt": "You are a helpful assistant with access to the following function:\n\n{\"name\": \"get_weather\", \"description\": \"Get current weather for a location\", \"parameters\": {\"type\": \"object\", \"properties\": {\"location\": {\"type\": \"string\", \"description\": \"City name\"}, \"unit\": {\"type\": \"string\", \"enum\": [\"celsius\", \"fahrenheit\"]}}, \"required\": [\"location\"]}}\n\nThe user asks: \"What's the weather like in San Francisco?\"\n\nRespond with a function call in JSON format using: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "arguments": {"type": "object", "additionalProperties": True}
            },
            "required": ["name", "arguments"],
            "additionalProperties": False
        },
        "expected_function": "get_weather",
        "expected_params_keys": ["location"]
    },
    {
        "id": "funcall_search_multi",
        "category": "function_calling",
        "difficulty": "medium",
        "prompt": "You are a helpful assistant with access to the following functions:\n\n1. {\"name\": \"search_web\", \"description\": \"Search the web for information\", \"parameters\": {\"type\": \"object\", \"properties\": {\"query\": {\"type\": \"string\"}, \"num_results\": {\"type\": \"integer\", \"default\": 10}}, \"required\": [\"query\"]}}\n\n2. {\"name\": \"send_email\", \"description\": \"Send an email\", \"parameters\": {\"type\": \"object\", \"properties\": {\"to\": {\"type\": \"string\"}, \"subject\": {\"type\": \"string\"}, \"body\": {\"type\": \"string\"}}, \"required\": [\"to\", \"subject\", \"body\"]}}\n\nThe user asks: \"Search for the best restaurants in NYC and email the results to john@example.com\"\n\nRespond with the appropriate function call(s) in JSON format. If multiple calls are needed, use an array. Use the format: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "arguments": {"type": "object", "additionalProperties": True}
                },
                "required": ["name", "arguments"],
                "additionalProperties": False
            },
            "minItems": 1,
            "maxItems": 2
        },
        "expected_functions": ["search_web", "send_email"],
        "expected_params_keys": [["query", "num_results"], ["to", "subject", "body"]]
    },
    {
        "id": "funcall_database_query",
        "category": "function_calling",
        "difficulty": "hard",
        "prompt": "You are a helpful assistant with access to the following function:\n\n{\"name\": \"query_database\", \"description\": \"Execute a SQL query on the database\", \"parameters\": {\"type\": \"object\", \"properties\": {\"query\": {\"type\": \"string\", \"description\": \"SQL query to execute\"}, \"database\": {\"type\": \"string\", \"enum\": [\"production\", \"staging\", \"analytics\"]}, \"limit\": {\"type\": \"integer\", \"default\": 100, \"maximum\": 1000}, \"format\": {\"type\": \"string\", \"enum\": [\"json\", \"csv\"], \"default\": \"json\"}}, \"required\": [\"query\", \"database\"]}}\n\nThe user asks: \"Get me the top 50 customers by revenue from the production database in CSV format\"\n\nRespond with a function call in JSON format using: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "arguments": {"type": "object", "additionalProperties": True}
            },
            "required": ["name", "arguments"],
            "additionalProperties": False
        },
        "expected_function": "query_database",
        "expected_params_keys": ["query", "database", "limit", "format"]
    }
]

# ============================================================
# TASK CATEGORY 4: Key-Value Extraction
# Extract structured info from unstructured text
# ============================================================

EXTRACTION_TASKS = [
    {
        "id": "extract_business_card",
        "category": "extraction",
        "difficulty": "easy",
        "prompt": "Extract the contact information from the following text into a JSON object with fields: name, phone, email, company, title.\n\nText: \"John Smith is a Senior Software Engineer at TechCorp Inc. You can reach him at john.smith@techcorp.com or call (555) 123-4567.\"",
        "expected_values": {
            "name": "John Smith",
            "phone": "(555) 123-4567",
            "email": "john.smith@techcorp.com",
            "company": "TechCorp Inc",
            "title": "Senior Software Engineer"
        },
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "phone": {"type": "string"},
                "email": {"type": "string"},
                "company": {"type": "string"},
                "title": {"type": "string"}
            },
            "required": ["name", "phone", "email", "company", "title"],
            "additionalProperties": False
        }
    },
    {
        "id": "extract_receipt",
        "category": "extraction",
        "difficulty": "medium",
        "prompt": "Extract receipt information from the following text into a JSON object with: store_name, date, items (array of objects with name and price), subtotal, tax, total.\n\nText: \"WALMART SUPERCENTER\nDate: 03/15/2026\nMilk 2% 1gal ........... $4.98\nSourdough Bread ........ $3.49\nOrganic Eggs 12ct ...... $5.99\nAvocados 3ct ........... $4.47\nSubtotal: $18.93\nTax (8.25%): $1.56\nTOTAL: $20.49\"",
        "expected_values": {
            "store_name": "WALMART SUPERCENTER",
            "date": "03/15/2026",
            "items": [
                {"name": "Milk 2% 1gal", "price": 4.98},
                {"name": "Sourdough Bread", "price": 3.49},
                {"name": "Organic Eggs 12ct", "price": 5.99},
                {"name": "Avocados 3ct", "price": 4.47}
            ],
            "subtotal": 18.93,
            "tax": 1.56,
            "total": 20.49
        },
        "schema": {
            "type": "object",
            "properties": {
                "store_name": {"type": "string"},
                "date": {"type": "string"},
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "price": {"type": "number"}
                        },
                        "required": ["name", "price"],
                        "additionalProperties": False
                    }
                },
                "subtotal": {"type": "number"},
                "tax": {"type": "number"},
                "total": {"type": "number"}
            },
            "required": ["store_name", "date", "items", "subtotal", "tax", "total"]
        }
    },
    {
        "id": "extract_api_log",
        "category": "extraction",
        "difficulty": "hard",
        "prompt": "Extract structured data from the following API log entry into JSON with: timestamp, method, path, status_code, response_time_ms, error (null if no error), and headers (object with content-type, x-request-id, user-agent).\n\nText: '[2026-05-05T10:23:45.678Z] POST /api/v2/users/authenticate -> 401 (145.3ms) | Headers: {\"content-type\": \"application/json\", \"x-request-id\": \"req-abc123def456\", \"user-agent\": \"MobileApp/3.2.1 (iOS 17.4)\"} | Error: Invalid credentials - email not verified'",
        "schema": {
            "type": "object",
            "properties": {
                "timestamp": {"type": "string"},
                "method": {"type": "string", "enum": ["GET", "POST", "PUT", "DELETE", "PATCH"]},
                "path": {"type": "string"},
                "status_code": {"type": "number"},
                "response_time_ms": {"type": "number"},
                "error": {"oneOf": [{"type": "string"}, {"type": "null"}]},
                "headers": {
                    "type": "object",
                    "properties": {
                        "content-type": {"type": "string"},
                        "x-request-id": {"type": "string"},
                        "user-agent": {"type": "string"}
                    },
                    "required": ["content-type", "x-request-id", "user-agent"],
                    "additionalProperties": False
                }
            },
            "required": ["timestamp", "method", "path", "status_code", "response_time_ms", "error", "headers"]
        }
    }
]

# ============================================================
# SYSTEM PROMPTS for different conditions
# ============================================================

SYSTEM_PROMPTS = {
    "basic": "You are a helpful assistant. Always respond with valid JSON. Do not include any text outside the JSON.",
    
    "structured": "You are a helpful assistant. When asked to generate structured output, you MUST respond with ONLY valid JSON. No markdown, no code blocks, no explanation - just the raw JSON. Ensure all required fields are present and types are correct.",
    
    "schema_given": "You are a helpful assistant. You will be given a JSON schema. Generate output that EXACTLY matches the schema. Respond with ONLY the JSON, nothing else. No markdown code fences. Validate your output mentally before responding."
}


In [ ]:
# --- Chat template formatter ---
def format_messages(messages):
    """Format messages using the model's chat template.
    Override for model-specific behavior."""
    # Default (Llama 3.2 Instruct uses the standard chat template):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


In [ ]:
# --- JSON extractor ---
def extract_json(raw):
    """Extract JSON from model output, handling markdown fences and extra text."""
    text = raw.strip()
    
    # 1. Direct parse
    try:
        return json.loads(text)
    except:
        pass
    
    # 2. Strip markdown fences (FIXED regex)
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass
    
    # 3. Find JSON boundaries
    for start_char, end_char in [('{', '}'), ('[', ']')]:
        s = text.find(start_char)
        e = text.rfind(end_char)
        if s != -1 and e > s:
            try:
                return json.loads(text[s:e+1])
            except:
                pass
    return None

In [ ]:
# --- Model-specific generate kwargs ---
# do_sample / temperature are set per-call inside run_single_task() based on the
# `temperature` argument (0.0 = greedy deterministic; >0 = sampling).
BASE_GENERATE_KWARGS = {
    "max_new_tokens": 512,
    "pad_token_id": 128001,  # Llama 3.2
}


In [ ]:
# --- Tokenizer/processor reference ---
# Set this to whichever you loaded
tok = tokenizer  # or tok = processor for Gemma

In [ ]:
def run_single_task(task, run_num=1, temperature=0.0):
    """Run a single benchmark task. Returns result dict.

    Args:
        task: task dict from task_definitions
        run_num: sample index within this (task, temperature) condition
        temperature: sampling temperature (0.0 = greedy deterministic)
    """

    # Build messages
    user_content = task["prompt"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS["structured"]},
        {"role": "user", "content": user_content},
    ]

    # Format with chat template
    text = format_messages(messages)

    # Tokenize
    inputs = tok(text=text, return_tensors="pt").to("cuda")

    # === CRITICAL: Calculate input_len AFTER tokenizing THIS task ===
    input_len = inputs["input_ids"].shape[1]

    # Configure generation kwargs for this temperature
    generate_kwargs = dict(BASE_GENERATE_KWARGS)
    generate_kwargs["do_sample"] = (temperature > 0.0)
    generate_kwargs["temperature"] = temperature
    if temperature > 0.0:
        # Let temperature govern sampling; keep helpers deterministic
        generate_kwargs["top_p"] = 1.0
        generate_kwargs["top_k"] = None

    # Generate
    start = time.time()
    outputs = model.generate(**inputs, **generate_kwargs)
    elapsed = (time.time() - start) * 1000

    # Decode ONLY the generated tokens
    response = tok.decode(outputs[0][input_len:], skip_special_tokens=True)
    num_tokens = outputs.shape[1] - input_len

    # Validation
    json_valid = False
    schema_valid = False
    parsed = None
    error_msg = None

    try:
        parsed = extract_json(response)
        if parsed is not None:
            json_valid = True
        else:
            error_msg = "Could not extract valid JSON"
    except Exception as e:
        error_msg = f"JSON parse error: {e}"

    if json_valid:
        try:
            import jsonschema
            jsonschema.validate(instance=parsed, schema=task["schema"])
            schema_valid = True
        except ImportError:
            # Fallback: check required fields
            required = task["schema"].get("required", [])
            if isinstance(parsed, dict):
                missing = [f for f in required if f not in parsed]
                if missing:
                    schema_valid = False
                    error_msg = f"Missing required fields: {missing}"
                else:
                    schema_valid = True
            else:
                schema_valid = True
        except Exception as e:
            schema_valid = False
            error_msg = f"Schema validation: {str(e)[:100]}"

    tokens_per_sec = round(num_tokens / (elapsed / 1000), 1) if elapsed > 0 else 0

    result = {
        "run": run_num,
        "temperature": temperature,
        "task_id": task["id"],
        "category": task["category"],
        "difficulty": task["difficulty"],
        "json_valid": json_valid,
        "schema_valid": schema_valid,
        "latency_ms": round(elapsed, 1),
        "tokens_generated": num_tokens,
        "tokens_per_sec": tokens_per_sec,
        "response_raw": response,
        "response_parsed": parsed,
        "error": error_msg,
    }

    # Print inline
    status = "\u2713" if schema_valid else ("~" if json_valid else "\u2717")
    print(f"  {status} {task['id']:<25} T={temperature:<4} JSON:{json_valid}  Schema:{schema_valid}  {elapsed:.0f}ms  {tokens_per_sec}tok/s")
    if error_msg:
        print(f"    Error: {error_msg[:80]}")

    return result


In [ ]:
# ============================================================
# PHASE 2: Temperature Robustness Probe
# 14 tasks x 3 temperatures (0.0, 0.3, 0.7) x 3 samples
# T=0.0 re-confirms Phase 1 (deterministic); T>0.0 tests sampling robustness.
# ============================================================

import csv
from collections import defaultdict
from pathlib import Path

ALL_TASKS = SIMPLE_JSON_TASKS + SCHEMA_ADHERENCE_TASKS + FUNCTION_CALLING_TASKS + EXTRACTION_TASKS

PHASE2_TEMPERATURES = [0.0, 0.3, 0.7]
PHASE2_SAMPLES = 3

results = []

total_runs = len(ALL_TASKS) * len(PHASE2_TEMPERATURES) * PHASE2_SAMPLES
print(f"PHASE 2 (PROBE): {MODEL_NAME}")
print(f"{len(ALL_TASKS)} tasks x {len(PHASE2_TEMPERATURES)} temps x {PHASE2_SAMPLES} samples = {total_runs} runs")
print(f"Started: {datetime.now().isoformat()}")
print("=" * 70)

for temperature in PHASE2_TEMPERATURES:
    print(f"\n--- Temperature {temperature} ---")
    for task in ALL_TASKS:
        for sample_num in range(1, PHASE2_SAMPLES + 1):
            result = run_single_task(task, run_num=sample_num, temperature=temperature)
            result["phase"] = 2
            result["model"] = MODEL_NAME
            results.append(result)

# --- Aggregate summary per (task, temperature) ---
agg = defaultdict(list)
for r in results:
    agg[(r["task_id"], r["temperature"])].append(r)

print(f"\n{'=' * 70}")
print(f"PHASE 2 SUMMARY: {MODEL_NAME}")
print(f"{'Task':<26}{'T':>5}{'N':>4}{'JSON':>9}{'Schema':>10}{'ms':>8}{'tok/s':>8}")
print("-" * 70)

by_temp = defaultdict(lambda: {"json": 0, "schema": 0, "n": 0, "lat": [], "tps": []})
for (task_id, temp), runs in sorted(agg.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    n = len(runs)
    jp = sum(r["json_valid"] for r in runs)
    sp = sum(r["schema_valid"] for r in runs)
    ams = sum(r["latency_ms"] for r in runs) / n
    atp = sum(r["tokens_per_sec"] for r in runs) / n
    print(f"{task_id:<26}{temp:>5.1f}{n:>4}{jp:>4}/{n:<4}{sp:>5}/{n:<4}{ams:>7.0f}{atp:>8.1f}")
    by_temp[temp]["json"] += jp
    by_temp[temp]["schema"] += sp
    by_temp[temp]["n"] += n
    by_temp[temp]["lat"].append(ams)
    by_temp[temp]["tps"].append(atp)

print("-" * 70)
print("OVERALL BY TEMPERATURE:")
for temp in PHASE2_TEMPERATURES:
    s = by_temp[temp]
    jp = 100 * s["json"] / s["n"]
    sp = 100 * s["schema"] / s["n"]
    al = sum(s["lat"]) / len(s["lat"])
    at = sum(s["tps"]) / len(s["tps"])
    print(f"  T={temp:<4}  JSON {jp:5.1f}%  Schema {sp:5.1f}%  {al:.0f}ms  {at:.1f}tok/s")

# Robustness interpretation
base = 100 * by_temp[0.0]["schema"] / by_temp[0.0]["n"]
t7 = 100 * by_temp[0.7]["schema"] / by_temp[0.7]["n"]
delta = base - t7
print(f"\nRobustness check (T=0.0 -> T=0.7 schema %): {base:.0f}% -> {t7:.0f}%  (delta {delta:+.0f}pp)")
if abs(delta) < 1:
    print("  -> Failures appear DETERMINISTIC under sampling.")
else:
    print("  -> Sampling induces new failures; consider expanding Phase 2.")

print(f"\nFinished: {datetime.now().isoformat()}")


In [ ]:
# ============================================================
# EXPORT: Save results
#   - CSV  : summary, one row per (task, temperature, sample)
#   - JSON : raw, preserves response_raw/response_parsed so a content-accuracy
#            metric can be computed LATER without re-running the model
# ============================================================

MODEL_SHORT = MODEL_NAME.split("/")[-1]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

Path("results/v2").mkdir(parents=True, exist_ok=True)

# 1. Raw JSON
raw_path = f"results/v2/phase2_{MODEL_SHORT}_raw_{timestamp}.json"
export = []
for r in results:
    er = dict(r)
    if er.get("response_parsed") is not None:
        er["response_parsed"] = str(er["response_parsed"])
    export.append(er)
with open(raw_path, "w") as f:
    json.dump(export, f, indent=2, default=str)
print(f"Raw results saved: {raw_path}")

# 2. CSV summary
csv_path = f"results/v2/phase2_{MODEL_SHORT}.csv"
CSV_COLUMNS = [
    "model", "phase", "temperature", "run", "task_id", "category", "difficulty",
    "json_valid", "schema_valid", "latency_ms", "tokens_generated",
    "tokens_per_sec", "error",
]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)
print(f"CSV results saved: {csv_path}  ({len(results)} rows)")


In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
import gc; gc.collect()